# CIFAR-10N Uncertainty Visualization with Decision Boundaries

**Features:**
- Small configurable subset sampling
- DINOv2 (768-dim) OR ResNet-10 (512-dim) feature extraction
- 2D decision boundary visualization via t-SNE/UMAP
- Aleatoric vs Epistemic uncertainty distinction

**The MLP is a fully connected network** operating on high-dimensional embeddings, creating non-linear decision boundaries.

In [ ]:
# ============================================================================
# CONFIGURATION - Change these parameters
# ============================================================================

FEATURE_EXTRACTOR = 'resnet10'  # 'dinov2' or 'resnet10'
UNDER_SUPPORTED_CLASSES = [0, 1]  # airplane, automobile
UNDER_TRAIN_PER_CLASS = 10
REGULAR_TRAIN_PER_CLASS = 50
EPOCHS = 20
MC_SAMPLES = 30
REDUCTION_METHOD = 'umap'  # 'tsne' or 'umap'
RANDOM_SEED = 42

print(f"✓ Config: {FEATURE_EXTRACTOR.upper()}, {EPOCHS} epochs, {MC_SAMPLES} MC samples")

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import models
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.manifold import TSNE
try:
    import umap
except:
    print("⚠️ UMAP not available, will use t-SNE")

sys.path.insert(0, str(Path.cwd()))
from src.data.cifar10n_loader import CIFAR10NDataset
from uq_classification.data_loader import sample_indices_for_fast_pilot
from uq_classification.models import EmbeddingDataset, EmbeddingDropoutMLP

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Load CIFAR-10N (reuses downloaded data)
dataset = CIFAR10NDataset(root='./cache/cifar10n', train=True, noise_type='worse_label', download=True)
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
print(f"Dataset: {len(dataset)} samples, {dataset.noise_mask.sum()} noisy")

In [ ]:
# Sample train/eval splits
split_spec = sample_indices_for_fast_pilot(
    dataset, under_supported_classes=UNDER_SUPPORTED_CLASSES,
    under_train_per_class=UNDER_TRAIN_PER_CLASS,
    regular_train_per_class=REGULAR_TRAIN_PER_CLASS,
    eval_per_group=50, seed=RANDOM_SEED
)
print(f"Train: {len(split_spec.train_indices)}, Eval: {len(split_spec.clean_eval_indices)+len(split_spec.aleatoric_eval_indices)+len(split_spec.epistemic_eval_indices)}")

In [ ]:
# Feature extractor
class ResNet10Extractor(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.feature_dim = 512
    def forward(self, x):
        return self.features(x).squeeze(-1).squeeze(-1)

def extract_features(dataset, indices, backbone_type, batch_size=32):
    subset = Subset(dataset, list(indices))
    loader = DataLoader(subset, batch_size=batch_size, shuffle=False)
    
    if backbone_type == 'dinov2':
        from src.models.dinov2_backbone import create_dinov2_model
        model = create_dinov2_model('dinov2_vits14', 10, 0.0, False, True).to(device)
        extract_fn = lambda imgs: model.extract_features(imgs.to(device))
        feat_dim = 768
    else:  # resnet10
        model = ResNet10Extractor().to(device)
        extract_fn = lambda imgs: model(imgs.to(device))
        feat_dim = 512
    
    model.eval()
    all_feats, all_labels, all_clean, all_noisy = [], [], [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Extracting {backbone_type}"):
            imgs, labels, clean, noisy = batch
            all_feats.append(extract_fn(imgs).cpu())
            all_labels.append(labels.cpu())
            all_clean.append(clean.cpu())
            all_noisy.append(noisy.cpu())
    
    return {
        'features': torch.cat(all_feats),
        'labels': torch.cat(all_labels),
        'clean_labels': torch.cat(all_clean),
        'is_noisy': torch.cat(all_noisy).bool()
    }

# Extract for all indices
all_idx = np.concatenate([split_spec.train_indices, split_spec.clean_eval_indices, 
                          split_spec.aleatoric_eval_indices, split_spec.epistemic_eval_indices])
feat_dict = extract_features(dataset, all_idx, FEATURE_EXTRACTOR)

n_train = len(split_spec.train_indices)
train_feats = feat_dict['features'][:n_train]
eval_feats = feat_dict['features'][n_train:]
feature_dim = train_feats.shape[1]
print(f"✓ Features: {feature_dim}-dim, train={train_feats.shape}, eval={eval_feats.shape}")

In [ ]:
# Train MLP classifier
train_ds = EmbeddingDataset(train_feats, feat_dict['labels'][:n_train], 
                             feat_dict['clean_labels'][:n_train], feat_dict['is_noisy'][:n_train],
                             torch.arange(n_train))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

model = EmbeddingDropoutMLP(feature_dim, 10, 256, 0.2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

losses = []
for epoch in tqdm(range(EPOCHS), desc="Training"):
    model.train()
    epoch_loss = 0
    for feats, labels in train_loader:
        feats, labels = feats.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(feats), labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(train_loader))
print(f"✓ Training complete, final loss: {losses[-1]:.4f}")

In [ ]:
# MC Dropout uncertainty
@torch.no_grad()
def mc_dropout(model, feats, n=30):
    model.eval()
    model.enable_dropout()
    all_probs = torch.stack([torch.softmax(model(feats.to(device)), 1) for _ in range(n)])
    mean_probs = all_probs.mean(0)
    pred_entropy = -(mean_probs * torch.log(mean_probs + 1e-10)).sum(1)
    expected_entropy = -(all_probs * torch.log(all_probs + 1e-10)).sum(2).mean(0)
    mutual_info = pred_entropy - expected_entropy
    return mean_probs, pred_entropy, mutual_info, expected_entropy

mean_probs, pred_ent, mutual_info, exp_ent = mc_dropout(model, eval_feats, MC_SAMPLES)
pred_classes = mean_probs.argmax(1)
print(f"✓ Uncertainty computed: entropy [{pred_ent.min():.3f}, {pred_ent.max():.3f}]")

In [ ]:
# Reduce to 2D for visualization
all_feats_np = torch.cat([train_feats, eval_feats]).cpu().numpy()

if REDUCTION_METHOD == 'umap':
    try:
        reducer = umap.UMAP(n_components=2, random_state=RANDOM_SEED)
        reduced = reducer.fit_transform(all_feats_np)
    except:
        print("⚠️ UMAP failed, using t-SNE")
        reduced = TSNE(n_components=2, random_state=RANDOM_SEED).fit_transform(all_feats_np)
else:
    reduced = TSNE(n_components=2, random_state=RANDOM_SEED).fit_transform(all_feats_np)

train_reduced = reduced[:len(train_feats)]
eval_reduced = reduced[len(train_feats):]
print(f"✓ Reduced to 2D using {REDUCTION_METHOD}")

In [ ]:
# Visualize uncertainty types
n_clean = len(split_spec.clean_eval_indices)
n_aleat = len(split_spec.aleatoric_eval_indices)
eval_type = torch.zeros(len(eval_feats), dtype=torch.long)
eval_type[:n_clean] = 0  # Clean
eval_type[n_clean:n_clean+n_aleat] = 1  # Aleatoric
eval_type[n_clean+n_aleat:] = 2  # Epistemic

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['green', 'orange', 'red']
labels = ['Clean', 'Aleatoric', 'Epistemic']

for i, (c, l) in enumerate(zip(colors, labels)):
    mask = eval_type == i
    axes[0].scatter(eval_reduced[mask, 0], eval_reduced[mask, 1], c=c, label=l, alpha=0.6, s=50)
axes[0].set_title(f'Uncertainty Types ({FEATURE_EXTRACTOR.upper()})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(eval_reduced[:, 0], eval_reduced[:, 1], 
                c=feat_dict['clean_labels'][n_train:].numpy(), cmap='tab10', alpha=0.6, s=50)
axes[1].set_title('True Classes')
axes[1].grid(True, alpha=0.3)

scatter = axes[2].scatter(eval_reduced[:, 0], eval_reduced[:, 1], 
                          c=pred_ent.cpu().numpy(), cmap='YlOrRd', alpha=0.6, s=50)
axes[2].set_title('Predictive Entropy')
plt.colorbar(scatter, ax=axes[2])
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Decision boundaries (approximate via nearest neighbor)
from scipy.spatial import cKDTree

x_min, x_max = reduced[:, 0].min() - 1, reduced[:, 0].max() + 1
y_min, y_max = reduced[:, 1].min() - 1, reduced[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
grid_pts = np.c_[xx.ravel(), yy.ravel()]

tree = cKDTree(reduced)
_, indices = tree.query(grid_pts, k=1)
grid_feats = torch.from_numpy(all_feats_np[indices]).float()

with torch.no_grad():
    model.eval()
    grid_logits = model(grid_feats.to(device))
    grid_probs = torch.softmax(grid_logits, 1)
    grid_preds = grid_probs.argmax(1).cpu().numpy()
    grid_conf = grid_probs.max(1)[0].cpu().numpy()

Z_class = grid_preds.reshape(xx.shape)
Z_conf = grid_conf.reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].contourf(xx, yy, Z_class, levels=np.arange(11)-0.5, cmap='tab10', alpha=0.3)
axes[0].scatter(eval_reduced[:, 0], eval_reduced[:, 1], 
                c=feat_dict['clean_labels'][n_train:].numpy(), cmap='tab10', s=100, edgecolors='black')
axes[0].set_title(f'Decision Boundaries ({FEATURE_EXTRACTOR.upper()})')
axes[0].grid(True, alpha=0.3)

axes[1].contourf(xx, yy, Z_conf, levels=20, cmap='RdYlGn', alpha=0.6)
for i, (c, l) in enumerate(zip(colors, labels)):
    mask = eval_type == i
    axes[1].scatter(eval_reduced[mask, 0], eval_reduced[mask, 1], c=c, label=l, s=100, edgecolors='black')
axes[1].set_title('Confidence Map')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("✓ Visualization complete!")

## Summary

This notebook demonstrated:
1. **Feature extraction** with DINOv2 (768-dim) or ResNet-10 (512-dim)
2. **MLP training** with dropout for uncertainty estimation
3. **MC Dropout** for epistemic/aleatoric decomposition
4. **2D visualization** of decision boundaries via t-SNE/UMAP

### Key Points:
- The MLP is a **fully connected network** (not convolutional)
- Decision boundaries are **non-linear** due to ReLU activation
- Visualization is approximate (high-D → 2D projection)
- Change `FEATURE_EXTRACTOR` to compare DINOv2 vs ResNet-10